
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Applying Agent Evaluation

## Overview

This lab provides hands-on experience with MLflow's evaluation framework for generative AI agents. You will work with the NYC Taxi dataset to build evaluation workflows using both built-in judges (correctness, safety) and custom guideline judges.

In this lab, you will create evaluation datasets, configure multiple types of judges, and analyze evaluation results to understand agent performance. The lab emphasizes practical implementation of automated evaluation workflows that can be applied to real-world AI applications.

## Learning Objectives

By the end of this lab, you will be able to:

- **Configure and execute** MLflow's built-in judges for correctness and safety assessment
- **Implement guideline judges** for custom business rule evaluation using natural language criteria
- **Create evaluation datasets** with proper structure for different judge types
- **Analyze evaluation results** using MLflow's tracing capabilities and result dataframes
- **Apply best practices** for separating evaluation models and managing configuration files


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;"> This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>


<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">That you might need to rerun the cells in the notebook if you receive an <strong>ValueError</strong> or <strong>MLflowException</strong> error regarding a datatype that was generated as a part of the agent's trace.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-4

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c
Resolved 144 packages in 3.35s
 Downloaded kiwisolver
 Downloaded hf-xet
 Downloaded zstandard
 Downloaded tiktoken
 Downloaded pydantic-core
 Downloaded tokenizers
 Downloaded grpcio
 Downloaded aiohttp
 Downloaded databricks-sdk
 Downloaded cryptography
 Downloaded sqlalchemy
 Downloaded pillow
 Downloaded fonttools
 Downloaded mlflow-tracing
 Downloaded matplotlib
 Downloaded numpy
 Downloaded pyarrow
 Downloaded scikit-learn
 Downloaded mlflow
 Downloaded pandas
 Downloaded litellm
 Downloaded botocore
 Downloaded scipy
 Downloaded openai
Prepared 130 packages in 10.81s
Uninstalled 106 packages in 77ms
Installed 130 packages in 1.58s
 - aiohappyeyeballs==2.4.4
 + aiohappyeyeballs==2.7.1
 - aiohttp==3.11.10
 + aiohttp==3.14.5
 - aiosignal==1.2.0
 + aiosignal==1.4.0
 + alembic==1.20.0
 - annotated-doc==0.0.4
 + annotated-doc==0.0.5
 - annotated-types==0.7.0
 + annotated

Pre-installed Databricks Runtime package(s) replaced: frozenlist: 1.5.0 -> 1.8.0, multidict: 6.1.0 -> 7.1.0, regex: 2024.11.6 -> 2026.9.29, packaging: 24.2 -> 25.0, jsonpointer: 3.0.0 -> 3.3, gitdb: 4.0.11 -> 4.0.12, MarkupSafe: 3.0.2 -> 3.0.4, importlib_metadata: 8.5.0 -> 8.9.0, starlette: 0.50.0 -> 1.9.0, huggingface_hub: 1.2.4 -> 2.2.0, jsonpatch: 1.33 -> 1.35, langsmith: 0.6.1 -> 0.14.11, jmespath: 1.0.1 -> 1.1.0, google-crc32c: 1.8.0 -> 1.9.0, uuid_utils: 0.12.0 -> 0.17.1, google-cloud-core: 2.5.0 -> 2.8.0, orjson: 3.11.5 -> 3.13.0, matplotlib: 3.10.0 -> 3.11.2, tokenizers: 0.22.2 -> 0.23.3, cycler: 0.11.0 -> 0.12.1, idna: 3.7 -> 3.20, cryptography: 44.0.1 -> 46.0.7, grpcio: 1.67.0 -> 1.67.1, scikit-learn: 1.6.1 -> 1.9.1, cachetools: 5.5.1 -> 6.2.6, pytz: 2024.1 -> 2026.5, mypy-extensions: 1.0.0 -> 1.1.0, tzdata: 2024.1 -> 2026.5, langchain-core: 1.2.6 -> 1.6.12, googleapis-common-protos: 1.65.0 -> 1.75.5, annotated-types: 0.7.0 -> 0.8.0, opentelemetry-sdk: 1.39.1 -> 1.45.1, scipy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c
Resolved 1 package in 59ms
Prepared 1 package in 66ms
Installed 1 package in 2ms
 + backoff==2.2.1
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c
Resolved 129 packages in 2.22s
 Downloaded openai-agents
 Downloaded mlflow-tracing
 Downloaded databricks-connect
 Downloaded mlflow-skinny
 Downloaded mlflow
Prepared 28 packages in 881ms
Uninstalled 11 packages in 50ms
Installed 29 packages in 227ms
 + aiohttp-retry==2.9.1
 - blinker==1.7.0
 ~ blinker==1.9.0
 + databricks-ai-bridge==0.22.0
 + databricks-ai-search==0.79
 - databricks-connect==18.1.9
 + databricks-conn

Pre-installed Databricks Runtime package(s) replaced: googleapis-common-protos: 1.65.0 -> 1.75.0, protobuf: 5.29.4 -> 5.29.6, databricks-connect: 18.1.9 -> 17.0.10, mlflow-skinny: 3.8.1 -> 3.17.1, PyJWT: 2.10.1 -> 2.15.1, wcwidth: 0.2.5 -> 0.9.2. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c
Resolved 43 packages in 770ms
Prepared 1 package in 130ms
Uninstalled 1 package in 0.46ms
Installed 1 package in 1ms
 - databricks-vectorsearch==0.75
 + databricks-vectorsearch==0.68
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Starting Demo Environment Setup
Using config: /Workspace/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Includes/config/config_4.yaml

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17704131_1792356850' found. Using it.

[2] Setting up schema...
  Catalog : labuser17704131_1792356850
  Schema  : demo_agent_eval
Refreshing environment

Demo Environment Setup Complete


2026/10/19 01:13:07 INFO mlflow.tracking.fluent: Experiment with name '/Users/labuser17704131_1792356850@vocareum.com/eval_lab_experiment' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/474748244217849', creation_time=1792372387969, effective_trace_archival_retention=None, experiment_id='474748244217849', last_update_time=1792372387969, lifecycle_stage='active', name='/Users/labuser17704131_1792356850@vocareum.com/eval_lab_experiment', tags={'mlflow.experiment.sourceName': '/Users/labuser17704131_1792356850@vocareum.com/eval_lab_experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'labuser17704131_1792356850@vocareum.com',
 'mlflow.ownerId': '71082494983411'}, trace_location=None, workspace='default'>

2026/10/19 01:13:19 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.17.1, required: mlflow==3.13.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Correctness Endpoint: databricks:/databricks-gpt-oss-20b
Safety Endpoint: databricks:/databricks-gpt-oss-20b
Guidelines Endpoint: databricks:/databricks-gpt-oss-120b


## B. Test Agent Functions

### B1. Test Agent Functions

Let's test the agent's capabilities by calling it with different types of queries to understand how it uses the available functions.

In [0]:
def agent_payload(question: str):
    return [{"input": [{"role": "user", "content": question}]}]

In [0]:
## Test the agent with a query about trip information
## Use a query that would trigger the get_trip_info function
# test_query_1 = <FILL_IN>
test_query_1 = "What's the average trip distance for trips from zip code 10001 to 10002?"
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ction. We'll query."}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ut 2.33\u202fmiles**.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Query: What's the average trip distance for trips from zip code 10001 to 10002?
Response: {'object': 'response', 'output': [{'type': 'function_call', 'id': 'chatcmpl_49cb8870-f61c-4946-a7b9-021d38bcfb72_1ec66bc3-ecc9-4ee8-80b0-be5fab4134f7', 'call_id': 'call_4f082f6a-9c3c-4d4b-8d9d-b1713096fd3e', 'name': 'labuser17704131_1792356850__demo_agent_eval__trip_info', 'arguments': '{"pickup_zip": 10001, "dropoff_zip": 10002}'}, {'type': 'function_call_output', 'call_id': 'call_4f082f6a-9c3c-4d4b-8d9d-b1713096fd3e', 'output': 'trips,avg_distance,avg_fare,min_fare,max_fare,total_fare,median_fare\n27,2.324814814814815,12.018518518518519,8.5,16.0,324.5,11.5\n'}, {'type': 'message', 'id': 'chatcmpl_d8a5f55a-79ae-47f6-a230-a38b93fc2907_2b6e847b-6ae2-4419-aa12-5923f4905c1c', 'content': [{'text': 'The average trip distance for trips from ZIP\u202f10001 to ZIP\u202f10002 is **about 2.33\u202fmiles**.', 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}]}


Trace(trace_id='tr-cee438caca9ffc9e3d72418357810140')

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task B1: Test Trip Info Query
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
test_query_1 = "What's the average trip distance for trips from zip code 10001 to 10002?"
result_1 = agent.predict(agent_payload(test_query_1))
print(f"Query: {test_query_1}")
print(f"Response: {result_1}")
```

</div>
</details>

In [0]:
## Test the agent with a fare calculation query
## Use a query that would trigger the calculate_fare_estimate function
# test_query_2 = <FILL_IN>
test_query_2 = "What would be the estimated fare for a 5.2 mile trip with a base fare of $3.50?"
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's.... Use function.\n\n'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...or a 5.2‑mile trip.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


Query: What would be the estimated fare for a 5.2 mile trip with a base fare of $3.50?
Response: {'object': 'response', 'output': [{'type': 'function_call', 'id': 'chatcmpl_72a9acb2-47b9-429b-9055-f040658723ad_003afcc6-334f-4685-9074-cb7c177299c5', 'call_id': 'call_066f1730-e55a-466b-bdb3-6b92bbbc506d', 'name': 'labuser17704131_1792356850__demo_agent_eval__calc_fare_est', 'arguments': '{"trip_distance":5.2,"base_fare":3.5}'}, {'type': 'function_call_output', 'call_id': 'call_066f1730-e55a-466b-bdb3-6b92bbbc506d', 'output': '25.87880880515136'}, {'type': 'message', 'id': 'chatcmpl_ac282feb-79c7-4d9c-9681-e3c362e4ce40_19027d9a-8d3a-4b3c-84f6-68e8e2a756d4', 'content': [{'text': '**Estimated Fare**: $25.88 (rounded to the nearest cent)\n\nThis estimate is calculated by adding the $3.50 base fare to the distance‑based rate derived from historical NYC taxi data for a 5.2‑mile trip.', 'type': 'output_text', 'annotations': []}], 'role': 'assistant'}]}


Trace(trace_id='tr-db7077430048d60842533f0f7ee0907b')

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task B1: Test Fare Calculation Query
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
test_query_2 = "What would be the estimated fare for a 5.2 mile trip with a base fare of $3.50?"
result_2 = agent.predict(agent_payload(test_query_2))
print(f"Query: {test_query_2}")
print(f"Response: {result_2}")
```

</div>
</details>

## C. Built-In Judges Implementation

### C1. Create Evaluation Dataset for Built-In Judges

Create an evaluation dataset with two inputs, expected facts, and expected responses for testing correctness and safety judges. Part of the evaluation dataset has been filled out for you.

**Hint:** If you get stuck, you can view a sample evaluation dataset in your volume at `/Volumes/{catalog_name}/{schema_name}/agent_vol/nyc_taxi_eval.json`.

In [0]:
## Create an evaluation dataset for built-in judges
## Include inputs and expectations with expected_facts for each query
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          # "role": <FILL_IN>,
          # "content": <FILL_IN>
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      # "expected_facts": [
      #   <FILL_IN>
      # ]
      "expected_facts": [
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average fare information"
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          # "role": <FILL_IN>,
          # "content": <FILL_IN>
          "role": "user",
          "content": "How many trips were taken from zip code 10003 to 10004?"
        }
      ]
    },
    "expectations": {
      # "expected_facts": [
      #   <FILL_IN>
      # ]
      "expected_facts": [
        "Pickup zip code is 10003",
        "Dropoff zip code is 10004",
        "Response includes trip count",
        "Response includes 'Are there any additional questions I can help with?'"
      ]
    }
  }
]

print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)

Created evaluation dataset for built-in judges:
[{'expectations': {'expected_facts': ['Pickup zip code is 10001',
                                      'Dropoff zip code is 10002',
                                      'Response includes average fare '
                                      'information']},
  'inputs': {'input': [{'content': 'What is the average fare for trips from '
                                   'zip code 10001 to 10002?',
                        'role': 'user'}]}},
 {'expectations': {'expected_facts': ['Pickup zip code is 10003',
                                      'Dropoff zip code is 10004',
                                      'Response includes trip count',
                                      "Response includes 'Are there any "
                                      'additional questions I can help '
                                      "with?'"]},
  'inputs': {'input': [{'content': 'How many trips were taken from zip code '
                             

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C1: Create Evaluation Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
eval_dataset_builtin = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10001",
        "Dropoff zip code is 10002",
        "Response includes average fare information"
      ]
    }
  },
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "How many trips were taken from zip code 10003 to 10004?"
        }
      ]
    },
    "expectations": {
      "expected_facts": [
        "Pickup zip code is 10003",
        "Dropoff zip code is 10004",
        "Response includes trip count",
        "Response includes 'Are there any additional questions I can help with?'"
      ]
    }
  }
]
print("Created evaluation dataset for built-in judges:")
from pprint import pprint
pprint(eval_dataset_builtin)
```

</div>
</details>

### C2. Configure Built-In Judges

Set up the Correctness and Safety judges using the configured endpoints.

In [0]:
## Import and configure the built-in judges
# from mlflow.genai.scorers import <FILL_IN>, <FILL_IN>
from mlflow.genai.scorers import Correctness, Safety

## Create correctness judge instance
# correctness_eval = <FILL_IN>(
#     model=<FILL_IN>
# )
correctness_eval = Correctness(
    model=correctness_eval_endpoint
)
## Create safety judge instance
# safety_eval = <FILL_IN>(
#     model=<FILL_IN>
# )
safety_eval = Safety(
    model=safety_endpoint
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C2: Configure Built-In Judges
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import Correctness, Safety
# Create correctness judge instance
correctness_eval = Correctness(
    model=correctness_eval_endpoint
)
# Create safety judge instance
safety_eval = Safety(
    model=safety_endpoint
)
```

</div>
</details>

### C3. Execute Built-In Judge Evaluation

Run the evaluation using both correctness and safety judges simultaneously. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with both built-in judges
# builtin_results = mlflow.genai.evaluate(
#     data=<FILL_IN>,
#     predict_fn=<FILL_IN>,
#     scorers=<FILL_IN>
# )
builtin_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, safety_eval]
)

2026/10/19 01:34:25 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/19 01:34:25 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]

2026/10/19 01:34:49 WARNING mlflow.genai.scorers.builtin_scorers: Failed to extract required fields from trace using LLM: Failed to parse response from judge model. Response: 
2026/10/19 01:34:54 WARNING mlflow.genai.scorers.builtin_scorers: Failed to extract required fields from trace using LLM: Failed to parse response from judge model. Response: 
2026/10/19 01:34:55 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'correctness': 1/2 failed, 'safety': 1/2 failed. Check individual trace assessments for detailed error messages.


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C3: Execute Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
builtin_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, safety_eval]
)
```

</div>
</details>

### C4. Analyze Built-In Judge Results

Examine the evaluation results and understand the scoring rationale by accessing `builtin_results.metrics` and `builtin_results.result_df`.

In [0]:
## Display the evaluation results and metrics
# print(f"Run ID: {<FILL_IN>}")
# print(f"Aggregated Metrics: {<FILL_IN>}")
# print("\nDetailed Results:")
# display(<FILL_IN>)

print(f"Run ID: {builtin_results.run_id}")
print(f"Aggregated Metrics: {builtin_results.metrics}")
print("\nDetailed Results:")
display(builtin_results.result_df)

Run ID: 340bdfb18dc54cb1baf1edb2166b6575
Aggregated Metrics: {'correctness/mean': np.float64(0.0), 'safety/mean': np.float64(1.0)}

Detailed Results:


trace_id expected_facts/value safety/value safety/error_message correctness/value correctness/error_message safety/rationale correctness/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-43c5d5870e8522d25e2bc14393d13655 null null Safety scorer requires the following fields: outputs. Provide them directly or pass a trace containing them. null Correctness scorer requires the following fields: outputs. Provide them directly or pass a trace containing them. null null {"info": {"trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "client_request_id": "tr-43c5d5870e8522d25e2bc14393d13655", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "474748244217849"}}, "request_time": "2026-10-19T01:34:46.286Z", "state": "ERROR", "trace_metadata": {"mlflow.sourceRun": "340bdfb18dc54cb1baf1edb2166b6575", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 365, \"output_tokens\": 82, \"total_tokens\": 447, \"cache_read_input_tokens\": 352}", "mlflow.modelId": "m-6ef3ca7b538449a5a6a50a0b1a4a76a2", "mlflow.databricks.notebookID": "4107611117782942", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.traceOutputs": "", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 26256, \"num_spans\": 4, \"max\": 10681, \"p25\": 4261, \"p50\": 5086, \"p75\": 6875}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792372121771_8607314988501489468_84fb7a386b3544e9869493434740a431", "mlflow.user": "spark-f21d573e-1d4e-4984-8773-a4", "mlflow.trace.sizeBytes": "26256", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-497dcab944b048359fd94944e3343a72", "assessment_name": "expected_facts", "trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "span_id": "5c307cc3adf34174", "source": {"source_type": "HUMAN", "source_id": "spark-f21d573e-1d4e-4984-8773-a4"}, "create_time": "2026-10-19T01:34:55.142Z", "last_update_time": "2026-10-19T01:34:55.142Z", "expectation": {"serialized_value": {"serialization_format": "JSON_FORMAT", "value": "[\"Pickup zip code is 10001\", \"Dropoff zip code is 10002\", \"Response includes average fare information\"]"}}, "metadata": {"mlflow.assessment.sourceRunId": "340bdfb18dc54cb1baf1edb2166b6575"}}, {"assessment_id": "a-8a8c5926289143808f21c3b24e92fe94", "assessment_name": "safety", "trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "span_id": "5c307cc3adf34174", "source": {"source_type": "CODE", "source_id": "safety"}, "create_time": "2026-10-19T01:34:54.943Z", "last_update_time": "2026-10-19T01:34:54.943Z", "feedback": {"value": null, "error": {"error_code": "SCORER_ERROR", "error_message": "Safety scorer requires the following fields: outputs. Provide them directly or pass a trace containing them.", "stack_trace": "Traceback (most recent call last):\n File \"/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/mlflow/genai/evaluation/harness.py\", line 940, in run_scorer\n value = call_with_ret

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task C4: Analyze Results
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
print(f"Run ID: {builtin_results.run_id}")
print(f"Aggregated Metrics: {builtin_results.metrics}")
print("\nDetailed Results:")
display(builtin_results.result_df)
```

</div>
</details>

### C5. Update and Iterate (Optional)

Did your agent pass your expectations? If not, try updating the system prompt in `Includes/artifacts/configs/nyc_taxi_agent_config.yaml` so that the agent aligns with your evaluation expectations. Iterate until all evaluations **Pass**.

For example, the default system prompt is

```
You are an expert in answering questions regarding NYC Taxi data.
```

Which isn't very descriptive if, for example, we pass the question _What is the average fare for trips from zip code 10001 to 10002?_ and expect the following in our output
- Pickup zip code is 10001,
- Dropoff zip code is 10002,
- Response includes average fare information,
- Response includes 'Are there any additional questions I can help with?'

then we might update our system prompt with _You are an expert in answering questions regarding NYC Taxi data. You will always respond with "Are there any additional questions I can help with?"_ (see screenshot below).

![update_system_prompt.png](https://files.training.databricks.com/binder/prod_main/genai-agent-evaluation-on-databricks-en_us-1.1.1/images/20261006T164734Z/Agent Evaluation on Databricks/Includes/images/applying agent eval/update_system_prompt.png "update_system_prompt.png")

After updating the prompt, you can redeploy your agent with the following code snippet.

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">CODE</span> Redeploy Agent with Updated Prompt
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow
import yaml
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

# Point to the rendered config file created by the setup orchestrator
CONFIG_YAML = "../Includes/artifacts/configs/nyc_taxi_agent_config.yaml"

# Load config from YAML
with open(CONFIG_YAML, "r") as f:
    cfg = yaml.safe_load(f)

# Pull values from YAML
CATALOG = cfg["CATALOG_NAME"]
SCHEMA = cfg["SCHEMA_NAME"]
MODEL_BASE_NAME = cfg.get("MODEL_NAME", "nyc_taxi")
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_BASE_NAME}"
LLM_ENDPOINT = cfg["LLM_ENDPOINT_NAME"]

# Build tool list from config
tool1 = cfg.get("TOOL1")
tool2 = cfg.get("TOOL2")
tool_names = [f"{CATALOG}.{SCHEMA}.{t}" for t in [tool1, tool2] if t]

# Build MLflow resources
resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT)] + [
    DatabricksFunction(function_name=t) for t in tool_names
]

# Agent source file (created by the setup orchestrator)
AGENT_PY = "../Includes/artifacts/nyc_taxi_agent.py"
ALIAS = "challenger"

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(tags={
    "training_type": "agent_eval_training",
    "model": LLM_ENDPOINT,
    "agent_type": "TOOL-CALLING",
}):
    logged = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model=AGENT_PY,
        model_config=CONFIG_YAML,
        artifacts={"agent_config": CONFIG_YAML},
        input_example={"input": [{"role": "user", "content": "hello"}]},
        pip_requirements=["databricks-openai", "backoff", "pyyaml"],
        resources=resources,
        registered_model_name=UC_MODEL_NAME,
    )

new_version = logged.registered_model_version
print("New UC version:", new_version)

from mlflow.tracking import MlflowClient
MlflowClient().set_registered_model_alias(
    name=UC_MODEL_NAME, alias=ALIAS, version=new_version
)

agent = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}@{ALIAS}")
```

</div>
</details>

You can now go back to sections **C3** and **C4** and re-evaluate.

## D. Guideline Judges Implementation

### D1. Create Global Guidelines Dataset

Create an evaluation dataset for testing global guidelines that apply uniformly to all responses. Since the `Guidelines` judge defines rules at the judge level (not per-row), the dataset only needs `inputs`.

In [0]:
## Create evaluation dataset for global guidelines
# eval_dataset_guidelines = [
#     {
#         "inputs": <FILL_IN>
#     }
# ]
eval_dataset_guidelines = [
    {
        "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    }
    }
]

print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)

Created evaluation dataset for global guidelines:
[{'inputs': {'input': [{'content': 'What is the average fare for trips from '
                                   'zip code 10001 to 10002?',
                        'role': 'user'}]}}]


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D1: Create Guidelines Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
eval_dataset_guidelines = [
  {
    "inputs": {
      "input": [
        {
          "role": "user",
          "content": "What is the average fare for trips from zip code 10001 to 10002?"
        }
      ]
    }
  }
]
print("Created evaluation dataset for global guidelines:")
pprint(eval_dataset_guidelines)
```

</div>
</details>

### D2. Implement Global Guidelines Judge

Create a global guidelines judge that ensures responses are professional and include specific formatting requirements.

In [0]:
## Import and configure the Guidelines judge
# from mlflow.genai.scorers import <FILL_IN>
from mlflow.genai.scorers import Guidelines

## Create a global guideline for professional responses
# professional_guideline = <FILL_IN>(
#     name=<FILL_IN>,
#     guidelines=[
#         <FILL_IN>,
#         <FILL_IN>
#     ],
#     model_name=<FILL_IN>
# )

professional_guideline = Guidelines(
    name="professional_response",
    guidelines=[
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ],
    model_name = guidelines_endpoint
)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D2: Implement Guidelines Judge
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from mlflow.genai.scorers import Guidelines
# Create a global guideline for professional responses
professional_guideline = Guidelines(
    name="professional_response",
    guidelines=[
        "The response should be professional and courteous",
        "The response should include specific numerical values when providing calculations or statistics",
        "The response should clearly indicate when it's using NYC taxi data"
    ],
    model_name=guidelines_endpoint
)
```

</div>
</details>

### D3. Execute Global Guidelines Evaluation

Run the evaluation using the global guidelines judge. Click on the **View evaluation results with MLflow** to inspect the results.

In [0]:
## Execute evaluation with global guidelines
# guidelines_results = mlflow.genai.evaluate(
#     data=<FILL_IN>,
#     predict_fn=<FILL_IN>,
#     scorers=<FILL_IN>
# )
guidelines_results = mlflow.genai.evaluate(
    data=eval_dataset_guidelines,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[professional_guideline]
)


2026/10/19 01:43:19 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/1 [Elapsed: 00:00, Remaining: ?]

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task D3: Execute Guidelines Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
guidelines_results = mlflow.genai.evaluate(
    data=eval_dataset_guidelines,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[professional_guideline]
)
```

</div>
</details>

## E. Comprehensive Evaluation Analysis

### E1. Compare Evaluation Results

Analyze and compare results from different judge types to understand their strengths and use cases.

In [0]:
## Display results from all evaluation runs for comparison
# print("=== BUILT-IN JUDGES RESULTS ===")
# print(f"Run ID: {<FILL_IN>}")
# print(f"Metrics: {<FILL_IN>}")
# display(<FILL_IN>)

# print("\n=== GLOBAL GUIDELINES RESULTS ===")
# print(f"Run ID: {<FILL_IN>}")
# print(f"Metrics: {<FILL_IN>}")
# display(<FILL_IN>)

print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {builtin_results.run_id}")
print(f"Metrics: {builtin_results.metrics}")
display(builtin_results.result_df)

print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {guidelines_results.run_id}")
print(f"Metrics: {guidelines_results.metrics}")
display(guidelines_results.result_df)

=== BUILT-IN JUDGES RESULTS ===
Run ID: 340bdfb18dc54cb1baf1edb2166b6575
Metrics: {'correctness/mean': np.float64(0.0), 'safety/mean': np.float64(1.0)}


trace_id expected_facts/value safety/value safety/error_message correctness/value correctness/error_message safety/rationale correctness/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-43c5d5870e8522d25e2bc14393d13655 null null Safety scorer requires the following fields: outputs. Provide them directly or pass a trace containing them. null Correctness scorer requires the following fields: outputs. Provide them directly or pass a trace containing them. null null {"info": {"trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "client_request_id": "tr-43c5d5870e8522d25e2bc14393d13655", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "474748244217849"}}, "request_time": "2026-10-19T01:34:46.286Z", "state": "ERROR", "trace_metadata": {"mlflow.sourceRun": "340bdfb18dc54cb1baf1edb2166b6575", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 365, \"output_tokens\": 82, \"total_tokens\": 447, \"cache_read_input_tokens\": 352}", "mlflow.modelId": "m-6ef3ca7b538449a5a6a50a0b1a4a76a2", "mlflow.databricks.notebookID": "4107611117782942", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.traceOutputs": "", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 26256, \"num_spans\": 4, \"max\": 10681, \"p25\": 4261, \"p50\": 5086, \"p75\": 6875}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792372121771_8607314988501489468_84fb7a386b3544e9869493434740a431", "mlflow.user": "spark-f21d573e-1d4e-4984-8773-a4", "mlflow.trace.sizeBytes": "26256", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-497dcab944b048359fd94944e3343a72", "assessment_name": "expected_facts", "trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "span_id": "5c307cc3adf34174", "source": {"source_type": "HUMAN", "source_id": "spark-f21d573e-1d4e-4984-8773-a4"}, "create_time": "2026-10-19T01:34:55.142Z", "last_update_time": "2026-10-19T01:34:55.142Z", "expectation": {"serialized_value": {"serialization_format": "JSON_FORMAT", "value": "[\"Pickup zip code is 10001\", \"Dropoff zip code is 10002\", \"Response includes average fare information\"]"}}, "metadata": {"mlflow.assessment.sourceRunId": "340bdfb18dc54cb1baf1edb2166b6575"}}, {"assessment_id": "a-8a8c5926289143808f21c3b24e92fe94", "assessment_name": "safety", "trace_id": "tr-43c5d5870e8522d25e2bc14393d13655", "span_id": "5c307cc3adf34174", "source": {"source_type": "CODE", "source_id": "safety"}, "create_time": "2026-10-19T01:34:54.943Z", "last_update_time": "2026-10-19T01:34:54.943Z", "feedback": {"value": null, "error": {"error_code": "SCORER_ERROR", "error_message": "Safety scorer requires the following fields: outputs. Provide them directly or pass a trace containing them.", "stack_trace": "Traceback (most recent call last):\n File \"/local_disk0/.ephemeral_nfs/envs/pythonEnv-f21d573e-1d4e-4984-8773-a4a38c4c522c/lib/python3.12/site-packages/mlflow/genai/evaluation/harness.py\", line 940, in run_scorer\n value = call_with_ret


=== GLOBAL GUIDELINES RESULTS ===
Run ID: 09d8b9df581a45b4bdf64b93491740c0
Metrics: {'professional_response/mean': np.float64(1.0)}


trace_id professional_response/value professional_response/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-5917b882a62c26977e00975fb373c30a yes The response is professional and courteous, addressing the user's question directly. It includes specific numerical values, providing an average fare of approximately $12.02, which satisfies the guideline for including numerical values in calculations or statistics. Additionally, the response clearly indicates that the data is based on aggregated historical data for trips between the specified ZIP codes, which implies the use of NYC taxi data. Therefore, all guidelines are satisfied. {"info": {"trace_id": "tr-5917b882a62c26977e00975fb373c30a", "client_request_id": "tr-5917b882a62c26977e00975fb373c30a", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "474748244217849"}}, "request_time": "2026-10-19T01:43:22.792Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "09d8b9df581a45b4bdf64b93491740c0", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 859, \"output_tokens\": 243, \"total_tokens\": 1102, \"cache_read_input_tokens\": 704}", "mlflow.modelId": "m-6ef3ca7b538449a5a6a50a0b1a4a76a2", "mlflow.databricks.notebookID": "4107611117782942", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.traceOutputs": "{\"tool_choice\": null, \"truncation\": null, \"id\": null, \"created_at\": null, \"error\": null, \"incomplete_details\": null, \"instructions\": null, \"metadata\": null, \"model\": null, \"object\": \"response\", \"output\": [{\"type\": \"function_call\", \"id\": \"chatcmpl_...", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 34206, \"num_spans\": 5, \"max\": 11779, \"p25\": 2782, \"p50\": 5507, \"p75\": 10615}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792372121771_8607314988501489468_84fb7a386b3544e9869493434740a431", "mlflow.user": "spark-f21d573e-1d4e-4984-8773-a4", "mlflow.trace.sizeBytes": "34206", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-4bbaa9d79072424fbeda67613fc9ec7d", "assessment_name": "professional_response", "trace_id": "tr-5917b882a62c26977e00975fb373c30a", "span_id": "d5475824b73f259e", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks"}, "create_time": "2026-10-19T01:43:28.799Z", "last_update_time": "2026-10-19T01:43:28.799Z", "feedback": {"value": "yes"}, "rationale": "The response is professional and courteous, addressing the user's question directly. It includes specific numerical values, providing an average fare of approximately $12.02, which satisfies the guideline for including numerical values in calculations or statistics. Additionally, the response clearly indicates that the data is based on aggregated historical data for trips between the specified ZIP codes, which implies the use of NYC taxi data. Therefore, all guidelines are satisfied.", "metadata": {"mlflow.assessment.sourceRunId": "09d8b9df581a45b4bdf64b93491740c0

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task E1: Compare Results
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
print("=== BUILT-IN JUDGES RESULTS ===")
print(f"Run ID: {builtin_results.run_id}")
print(f"Metrics: {builtin_results.metrics}")
display(builtin_results.result_df)
print("\n=== GLOBAL GUIDELINES RESULTS ===")
print(f"Run ID: {guidelines_results.run_id}")
print(f"Metrics: {guidelines_results.metrics}")
display(guidelines_results.result_df)
```

</div>
</details>

### E2. Create Comprehensive Evaluation

Combine multiple judge types in a single evaluation run to get comprehensive quality assessment.

In [0]:
## Create a comprehensive evaluation using multiple judge types
## Use the built-in dataset and combine correctness with global guidelines
# comprehensive_results = mlflow.genai.evaluate(
#     data=<FILL_IN>,
#     predict_fn=<FILL_IN>,
#     scorers=<FILL_IN>
# )

# print("=== COMPREHENSIVE EVALUATION RESULTS ===")
# print(f"Run ID: {<FILL_IN>}")
# print(f"Metrics: {<FILL_IN>}")
# display(<FILL_IN>)

comprehensive_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, professional_guideline]
)

print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {comprehensive_results.run_id}")
print(f"Metrics: {comprehensive_results.metrics}")
display(comprehensive_results.result_df)

2026/10/19 01:44:11 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/2 [Elapsed: 00:00, Remaining: ?]

=== COMPREHENSIVE EVALUATION RESULTS ===
Run ID: 6d2df6bc69bb4eaebf710afb4caa5797
Metrics: {'correctness/mean': np.float64(0.5), 'professional_response/mean': np.float64(1.0)}


trace_id expected_facts/value correctness/value correctness/rationale professional_response/value professional_response/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-be3776322b1bf78ad3e0396a3abf542c null yes The document explicitly states that the data concerns trips with pickup ZIP 10001 and drop‑off ZIP 10002, and it provides an average fare of $12.02. Therefore, the claim that the pickup zip code is 10001, the dropoff zip code is 10002, and the response includes average fare information is fully supported. The response is correct. yes The response is professional and courteous, addressing the user's question appropriately. It includes specific numerical values for the average fare, number of trips, and other relevant metrics, which satisfies the guideline for providing calculations or statistics. Additionally, it clearly indicates that the data is from NYC taxi data, fulfilling that guideline as well. Since all guidelines are satisfied, the result is 'yes'. {"info": {"trace_id": "tr-be3776322b1bf78ad3e0396a3abf542c", "client_request_id": "tr-be3776322b1bf78ad3e0396a3abf542c", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "474748244217849"}}, "request_time": "2026-10-19T01:44:14.497Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "6d2df6bc69bb4eaebf710afb4caa5797", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 856, \"output_tokens\": 365, \"total_tokens\": 1221, \"cache_read_input_tokens\": 736}", "mlflow.modelId": "m-6ef3ca7b538449a5a6a50a0b1a4a76a2", "mlflow.databricks.notebookID": "4107611117782942", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.7 Lab - Applying Agent Evaluation", "mlflow.traceOutputs": "{\"tool_choice\": null, \"truncation\": null, \"id\": null, \"created_at\": null, \"error\": null, \"incomplete_details\": null, \"instructions\": null, \"metadata\": null, \"model\": null, \"object\": \"response\", \"output\": [{\"type\": \"function_call\", \"id\": \"chatcmpl_...", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 34490, \"num_spans\": 5, \"max\": 11943, \"p25\": 2780, \"p50\": 5498, \"p75\": 10751}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792372121771_8607314988501489468_84fb7a386b3544e9869493434740a431", "mlflow.user": "spark-f21d573e-1d4e-4984-8773-a4", "mlflow.trace.sizeBytes": "34490", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-da68e7dee17b40aabb1aa061092d0f47", "assessment_name": "expected_facts", "trace_id": "tr-be3776322b1bf78ad3e0396a3abf542c", "span_id": "11d75d1c9437c807", "source": {"source_type": "HUMAN", "source_id": "spark-f21d573e-1d4e-4984-8773-a4"}, "create_time": "2026-10-19T01:44:22.542Z", "last_update_time": "2026-10-19T01:44:22.542Z", "expectation": {"serialized_value": {"serialization_format": "JSON_FORMAT", "value": "[\"Pickup zip code is 10001\", \"Dropoff zip code is 10002\", \"Response includes average fare information\"]"}}, "metadata": {"mlflow.assessment.sourceRunId": "6d2df6bc69bb

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task E2: Comprehensive Evaluation
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
comprehensive_results = mlflow.genai.evaluate(
    data=eval_dataset_builtin,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval, professional_guideline]
)
print("=== COMPREHENSIVE EVALUATION RESULTS ===")
print(f"Run ID: {comprehensive_results.run_id}")
print(f"Metrics: {comprehensive_results.metrics}")
display(comprehensive_results.result_df)
```

</div>
</details>

## F. Conclusion

In this lab, you have successfully implemented a comprehensive evaluation framework for AI agents using MLflow's built-in and guideline judges. You learned how to:

- Configure and execute multiple types of judges for different evaluation needs
- Create properly structured evaluation datasets for various judge types
- Analyze evaluation results using MLflow's tracing and result analysis capabilities
- Apply best practices for evaluation configuration and model management

These evaluation techniques provide the foundation for ensuring your AI agents meet quality standards and perform reliably in production environments. The combination of objective built-in judges and flexible guideline judges enables comprehensive quality assessment that can adapt to your specific business requirements.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
